<a href="https://colab.research.google.com/github/ksehar99/DL_Blockchain_Pipeline_For_DR_EarlyDiagnosis/blob/main/Copy_of_DR_DL_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sub-Pipeline 2 - Deep Learning Model
## Diabetic Retinopathy Classification · EfficientNetB3 · Transfer Learning · APTOS 2019

In [2]:
# ─── Setup & Imports ────────────────────────────────────────────────────────

# Mount Google Drive — dataset and patient JSON are stored persistently here
from google.colab import drive
drive.mount('/content/drive')

# Kaggle API credentials — required to download the APTOS 2019 dataset
import os
os.environ['KAGGLE_USERNAME'] = 'khadija4058'
os.environ['KAGGLE_KEY'] = 'KGAT_61e373e830995974eee3022ca7f6673f'

# Dataset download — run only once, stored in Drive to persist across sessions
# !kaggle datasets download -d mariaherrerot/aptos2019
# !unzip aptos2019.zip -d /content/drive/MyDrive/aptos2019

# Install required libraries
!pip install web3    # For connecting to Ethereum blockchain via Web3.py
!pip install faker   # For generating synthetic patient data

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.4/51.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.0/344.0 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.0/176.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 33.1 MB/s eta 0:00:00


In [3]:
# ─── Imports ─────────────────────────────────────────────────────────────────

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array, array_to_img
from sklearn.utils.class_weight import compute_class_weight

# ─── Dataset Setup ───────────────────────────────────────────────────────────

# Verify dataset structure and image count
print("Dataset contents:", os.listdir('/content/drive/MyDrive/aptos2019'))
print("Total training images:", len(os.listdir('/content/drive/MyDrive/aptos2019/train_images/train_images')))

# Load training labels CSV
# Columns: id_code (image filename), diagnosis (DR severity 0-4)
df = pd.read_csv('/content/drive/MyDrive/aptos2019/train_1.csv')
print(df.head())
print("Dataset shape:", df.shape)

Dataset contents: ['test.csv', 'train_1.csv', 'valid.csv', 'test_images', 'train_images', 'val_images']
Total training images: 2930
        id_code  diagnosis
0  1ae8c165fd53          2
1  1b329a127307          1
2  1b32e1d775ea          4
3  1b3647865779          0
4  1b398c0494d1          0
Dataset shape: (2930, 2)


In [4]:
# ─── Model Architecture ───────────────────────────────────────────────────────

# Load EfficientNetB3 pretrained on ImageNet as the base feature extractor
# include_top=False — removes the original 1000-class classification head
# weights='imagenet' — loads pretrained weights instead of training from scratch
base_model = EfficientNetB3(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model — preserve pretrained ImageNet knowledge
# Only the custom classification head will be trained initially
base_model.trainable = False

# Add custom classification head for DR severity (5 classes)
features = base_model.output
pooled_features = GlobalAveragePooling2D()(features)   # Compress spatial features
dropped_features = Dropout(0.3)(pooled_features)        # Prevent overfitting
output = Dense(5, activation='softmax')(dropped_features) # 5-class probability output

# Assemble full model
model = Model(inputs=base_model.input, outputs=output)

# Compile — sparse_categorical_crossentropy used because labels are integers (0-4)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
# ─── Class Imbalance & Data Preparation ──────────────────────────────────────

# Visualize class distribution — dataset is heavily imbalanced
print("Class distribution:\n", df['diagnosis'].value_counts())

# ─── Class Weights ────────────────────────────────────────────────────────────
# Weights reflect both data imbalance and clinical importance.
# Early-stage DR (Mild, Moderate) gets higher weight for early diagnosis goal.
class_weight_dict = {
    0: 1.0,   # No DR — most common, baseline weight
    1: 3.0,   # Mild — early diagnosis priority
    2: 3.0,   # Moderate — early diagnosis priority
    3: 2.0,   # Severe
    4: 2.0    # Proliferative
}

# ─── Augmentation Planning ────────────────────────────────────────────────────
# Calculate how many images need to be generated per minority class
# Target: ~600 images per class to reduce imbalance
def augment_class(class_label, target_count):
    """Prints how many augmented images are needed for a given class."""
    current_count = len(df[df['diagnosis'] == class_label])
    needed = target_count - current_count
    print(f"Class {class_label}: {current_count} images, {needed} needed")

augment_class(1, 600)
augment_class(3, 600)
augment_class(4, 600)

# ─── Load Augmented Dataset ───────────────────────────────────────────────────
# Augmented images were pre-generated and saved to the training folder.
# Reload CSV and include augmented images in the dataframe.

# Reload CSV with .png extension added to filenames
df = pd.read_csv('/content/drive/MyDrive/aptos2019/train_1.csv')
df['id_code'] = df['id_code'] + '.png'
df['diagnosis'] = df['diagnosis'].astype(str)

# Collect augmented image filenames (prefixed with 'aug_')
aug_files = [f for f in os.listdir('/content/drive/MyDrive/aptos2019/train_images/train_images') if f.startswith('aug_')]

aug_rows = []
for f in aug_files:
    class_label = f.split('_')[1]   # Extract class from filename: aug_1_58.png → 1
    aug_rows.append({'id_code': f, 'diagnosis': class_label})

# Merge original and augmented data
aug_df = pd.DataFrame(aug_rows)
df = pd.concat([df, aug_df], ignore_index=True)

# Shuffle to mix original and augmented samples
df = df.sample(frac=1).reset_index(drop=True)

print("Updated class distribution:\n", df['diagnosis'].value_counts())
print(f"Total images: {len(df)}")

Class distribution:
 diagnosis
0    1434
2     808
1     300
4     234
3     154
Name: count, dtype: int64
Class 1: 300 images, 300 needed
Class 3: 154 images, 446 needed
Class 4: 234 images, 366 needed
Updated class distribution:
 diagnosis
0    1434
2     808
1     300
4     234
3     154
Name: count, dtype: int64
Total images: 2930


In [6]:
# ─── Data Generators ──────────────────────────────────────────────────────────

# Training generator — includes on-the-fly augmentation to improve generalization
# Augmentation simulates real-world variations in retinal image capture
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # EfficientNet-specific normalization
    rotation_range=360,        # Retinal images can be captured at any angle
    horizontal_flip=True,      # Mirror image — valid variation for retinas
    vertical_flip=True,        # Vertical mirror — valid for fundus images
    zoom_range=0.1,            # Slight zoom variation
    brightness_range=[0.8, 1.2], # Simulate lighting differences
    validation_split=0.2       # 80/20 train-validation split
)

# Validation generator — no augmentation, only preprocessing
# Validation should reflect real data — no artificial transformations
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

# Training data pipeline — feeds batches of 16 images during training
train_generator = train_datagen.flow_from_dataframe(
    dataframe=df,
    directory='/content/drive/MyDrive/aptos2019/train_images/train_images',
    x_col='id_code',
    y_col='diagnosis',
    target_size=(224, 224),   # EfficientNetB3 input size
    batch_size=16,
    class_mode='sparse',      # Labels are integers (0-4), not one-hot
    subset='training'
)

# Validation data pipeline — evaluates model after each epoch
val_generator = val_datagen.flow_from_dataframe(
    dataframe=df,
    directory='/content/drive/MyDrive/aptos2019/train_images/train_images',
    x_col='id_code',
    y_col='diagnosis',
    target_size=(224, 224),
    batch_size=16,
    class_mode='sparse',
    subset='validation'
)

Found 2344 validated image filenames belonging to 5 classes.
Found 586 validated image filenames belonging to 5 classes.


In [ ]:
# ─── Training: Phase 1 — Head Only ───────────────────────────────────────────
# Base model is frozen. Only the custom classification head is trained.
# This allows the model to learn DR-specific features without disturbing
# the pretrained ImageNet weights.
# NOTE: Phase 1 training is complete — cells are commented out to prevent
#       accidental retraining which would overwrite the saved model.

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# history1 = model.fit(
#     train_generator,
#     epochs=8,
#     validation_data=val_generator,
#     class_weight=class_weight_dict,
#     callbacks=[
#         tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
#         tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7)
#     ]
# )

# # Save Phase 1 model to Drive
# model.save('/content/drive/MyDrive/dr_model_phase1.h5')
# print("Phase 1 model saved!")

# Load Phase 1 model — use this if session was reset
model = tf.keras.models.load_model('/content/drive/MyDrive/dr_model_phase1.h5')
print("Phase 1 model loaded!")

Epoch 1/8
147/147 ━━━━━━━━━━━━━━━━━━━━ 481s 3s/step - accuracy: 0.6510 - loss: 2.0427 - val_accuracy: 0.7082 - val_loss: 0.8484 - learning_rate: 0.0010
Epoch 2/8
147/147 ━━━━━━━━━━━━━━━━━━━━ 370s 3s/step - accuracy: 0.7235 - loss: 1.6784 - val_accuracy: 0.7457 - val_loss: 0.7391 - learning_rate: 0.0010
Epoch 3/8
147/147 ━━━━━━━━━━━━━━━━━━━━ 367s 3s/step - accuracy: 0.7376 - loss: 1.5794 - val_accuracy: 0.7509 - val_loss: 0.7148 - learning_rate: 0.0010
Epoch 4/8
147/147 ━━━━━━━━━━━━━━━━━━━━ 357s 2s/step - accuracy: 0.7449 - loss: 1.5433 - val_accuracy: 0.7235 - val_loss: 0.7823 - learning_rate: 0.0010
Epoch 5/8
147/147 ━━━━━━━━━━━━━━━━━━━━ 360s 2s/step - accuracy: 0.7526 - loss: 1.4958 - val_accuracy: 0.7509 - val_loss: 0.7072 - learning_rate: 0.0010
Epoch 6/8
147/147 ━━━━━━━━━━━━━━━━━━━━ 357s 2s/step - accuracy: 0.7658 - loss: 1.4475 - val_accuracy: 0.7577 - val_loss: 0.6750 - learning_rate: 0.0010
Epoch 7/8
147/147 ━━━━━━━━━━━━━━━━━━━━ 360s 2s/step - accuracy: 0.7619 - loss: 1.4401 - 

In [7]:
import tensorflow as tf

def focal_loss(gamma=2.0, alpha=0.25):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_one_hot = tf.one_hot(y_true, depth=5)
        ce = -y_true_one_hot * tf.math.log(y_pred + 1e-7)
        weight = alpha * y_true_one_hot * tf.pow(1 - y_pred, gamma)
        focal = weight * ce
        return tf.reduce_mean(tf.reduce_sum(focal, axis=-1))
    return loss_fn

In [8]:
# # ─── Training: Phase 2 — Fine Tuning ─────────────────────────────────────────
# Last 30 layers of EfficientNetB3 are unfrozen for fine-tuning.
# Lower learning rate (1e-5) used to avoid destroying pretrained weights.
# NOTE: Phase 2 training is complete — cells are commented out to prevent
#       accidental retraining which would overwrite the saved model.

# Load Phase 1 model — use this if session was reset
model = tf.keras.models.load_model('/content/drive/MyDrive/dr_model_phase1.h5')
print("Phase 1 model loaded!")

# Unfreeze last 30 layers of base model for fine-tuning
for layer in base_model.layers[-30:]:
    layer.trainable = True

# Recompile with lower learning rate and focal loss for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss= focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7)
    ]
)

# Save final fine-tuned model to Drive
model.save('/content/drive/MyDrive/dr_model_phase2.h5')
print("Phase 2 model saved!")

Phase 1 model loaded!
Epoch 1/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 659s 4s/step - accuracy: 0.7760 - loss: 0.1714 - val_accuracy: 0.7935 - val_loss: 0.0818 - learning_rate: 1.0000e-05
Epoch 2/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 393s 3s/step - accuracy: 0.7743 - loss: 0.1683 - val_accuracy: 0.7918 - val_loss: 0.0785 - learning_rate: 1.0000e-05
Epoch 3/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 392s 3s/step - accuracy: 0.7722 - loss: 0.1615 - val_accuracy: 0.7867 - val_loss: 0.0774 - learning_rate: 1.0000e-05
Epoch 4/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 395s 3s/step - accuracy: 0.7675 - loss: 0.1553 - val_accuracy: 0.7884 - val_loss: 0.0768 - learning_rate: 1.0000e-05
Epoch 5/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 399s 3s/step - accuracy: 0.7790 - loss: 0.1571 - val_accuracy: 0.7833 - val_loss: 0.0763 - learning_rate: 1.0000e-05
Epoch 6/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 401s 3s/step - accuracy: 0.7701 - loss: 0.1570 - val_accuracy: 0.7850 - val_loss: 0.0760 - learning_rate: 1.0000e-05
Epoch 7/10
147/147 ━━━━━━━━━━━━━━━━━

Phase 2 model saved!


In [9]:
# ─── Model Evaluation ────────────────────────────────────────────────────────
# Standalone cell — can be run independently after a session reset.
# Loads the final fine-tuned model and evaluates it on the validation set.

import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report
from google.colab import drive

# Mount Drive — skips if already mounted
drive.mount('/content/drive', force_remount=False)

# Test data se evaluation
df_test = pd.read_csv('/content/drive/MyDrive/aptos2019/test.csv')
df_test['id_code'] = df_test['id_code'] + '.png'
df_test['diagnosis'] = df_test['diagnosis'].astype(str)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=df_test,
    directory='/content/drive/MyDrive/aptos2019/test_images/test_images',
    x_col='id_code',
    y_col='diagnosis',
    target_size=(224, 224),
    batch_size=32,
    class_mode='sparse',
    shuffle=False
)

test_generator.reset()
y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.labels

print(classification_report(
    y_true,
    y_pred_classes,
    target_names=['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 366 validated image filenames belonging to 5 classes.
12/12 ━━━━━━━━━━━━━━━━━━━━ 215s 17s/step
               precision    recall  f1-score   support

        No DR       0.95      0.95      0.95       199
         Mild       0.34      0.37      0.35        30
     Moderate       0.62      0.84      0.71        87
       Severe       0.44      0.24      0.31        17
Proliferative       0.50      0.12      0.20        33

     accuracy                           0.77       366
    macro avg       0.57      0.50      0.50       366
 weighted avg       0.76      0.77      0.75       366



In [11]:
# ─── Model Evaluation with Test Data + TTA ───────────────────────────────────
# Evaluates the final model on unseen test data using Test Time Augmentation.
# TTA averages predictions over multiple augmented versions of each image
# to improve recall on minority classes without retraining.

import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from sklearn.metrics import classification_report
from google.colab import drive

# Mount Drive
drive.mount('/content/drive', force_remount=False)

# Load model
model = tf.keras.models.load_model(
    '/content/drive/MyDrive/dr_model_phase2.h5',
    compile=False
)
print("Model loaded!")

# Load test CSV
df_test = pd.read_csv('/content/drive/MyDrive/aptos2019/test.csv')
TEST_IMAGE_DIR = '/content/drive/MyDrive/aptos2019/test_images/test_images'

CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

# ─── TTA Function ────────────────────────────────────────────────────────────

def predict_with_tta(image_path, model, n_augments=5):
    """
    Runs inference on one image using TTA.
    Averages predictions across original + n augmented versions.
    """
    img_array = img_to_array(load_img(image_path, target_size=(224, 224)))

    predictions = []

    # Original prediction
    orig = preprocess_input(np.expand_dims(img_array, axis=0))
    predictions.append(model.predict(orig, verbose=0))

    # Augmented predictions
    datagen = ImageDataGenerator(
        horizontal_flip=True,
        vertical_flip=True,
        rotation_range=30,
        zoom_range=0.1
    )
    for _ in range(n_augments):
        aug = preprocess_input(
            next(datagen.flow(np.expand_dims(img_array, axis=0)))
        )
        predictions.append(model.predict(aug, verbose=0))

    return int(np.argmax(np.mean(predictions, axis=0)))

# ─── Run Evaluation ──────────────────────────────────────────────────────────

y_true = []
y_pred = []

total = len(df_test)
for i, (_, row) in enumerate(df_test.iterrows()):
    image_path = f"{TEST_IMAGE_DIR}/{row['id_code']}.png"
    true_label = int(row['diagnosis'])
    pred_label = predict_with_tta(image_path, model)
    y_true.append(true_label)
    y_pred.append(pred_label)
    print(f"Progress: {i+1}/{total}", end='\r')

print("\nClassification Report (Test Data + TTA):")
print(classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES
))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model loaded!
Progress: 366/366
Classification Report (Test Data + TTA):
               precision    recall  f1-score   support

        No DR       0.98      0.98      0.98       199
         Mild       0.43      0.43      0.43        30
     Moderate       0.62      0.85      0.72        87
       Severe       0.43      0.18      0.25        17
Proliferative       0.55      0.18      0.27        33

     accuracy                           0.80       366
    macro avg       0.60      0.52      0.53       366
 weighted avg       0.79      0.80      0.78       366

